In [39]:
import os

from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from dotenv import load_dotenv
from openai import AsyncOpenAI


In [40]:
load_dotenv(override=True)

True

In [41]:
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL")
deepseek_model = os.getenv("DEEPSEEK_MODEL")

token_harbor_api_key = os.getenv("TOKEN_HARBOR_API_KEY")
token_harbor_base_url = os.getenv("TOKEN_HARBOR_BASE_URL")
token_harbor_model = os.getenv("TOKEN_HARBOR_MODEL")

In [42]:
instruction = """
You are a sales agent working for Finance,
a company that provides a SaaS tool for ensuring compliance with the GDPR.
You write compelling emails to customers to help them comply with the GDPR.
"""

In [43]:
deepseek_client = AsyncOpenAI(
    api_key=deepseek_api_key,
    base_url=deepseek_base_url
)

harbor_client = AsyncOpenAI(
    api_key=token_harbor_api_key,
    base_url=token_harbor_base_url
)

In [44]:
deepseek_chat_model = OpenAIChatCompletionsModel(model=deepseek_model, openai_client=deepseek_client)
harbor_chat_model = OpenAIChatCompletionsModel(model=token_harbor_model, openai_client=harbor_client)

In [45]:
sales_agent1 = Agent(name="SalesAgent1", instructions=instruction, model=deepseek_chat_model)
sales_agent2 = Agent(name="SalesAgent2", instructions=instruction, model=harbor_chat_model)
sales_agent3 = Agent(name="SalesAgent3", instructions=instruction, model=harbor_chat_model)

In [46]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [47]:
from messeger import send_email, push


def send_message(subject, text_body, html_body):
    send_email(subject, text_body, html_body)
    push(text_body)

In [48]:
send_message("Yet another test", "This is a test", "<p>This is a test</p>")

Pushing message to Pushover: This is a test


In [49]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """Send an email with the given subject and body to the specified recipient."""
    send_email(subject, text_body, html_body)
    return f"Email sent to {os.getenv('EMAIL_RECIPIENT')}"

In [50]:
tools = [tool1, tool2, tool3, send_email_tool]

In [51]:
instruction = """
You are a Sales Manager at ComplAI. your gold is to find the single cold sales email using the sales_writer tools.
"""

tasks = """
Follow these steps

1. Generate Drafts: use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write s sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and select: review the drafts and choose the single best email using your judgement of which one is most effective.

3. Use your tool to send the best email (and only the best email) to the use. only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instruction, model=deepseek_chat_model, tools=tools)


In [52]:
with trace("Sales Manager across different model"):
    result = await Runner.run(sales_manager, tasks)
print(result.final_output)

Done. ✅

**Summary:**
1. **Generated** three distinct drafts using the three sales email writers.
2. **Selected** the strongest one — Draft 2 — for being the most complete and effective (preview text, concrete value props, risk quantification, clear CTA, fallback offer, and unsubscribe compliance).
3. **Sent** exactly one email with the subject "GDPR compliance without the spreadsheet chaos."

Only the single best email was sent, as instructed.


In [53]:
from pydantic import BaseModel, Field


class EmailReview(BaseModel):
    is_professional: bool = Field(description="Indicates whether the email is professional in tone and content.")
    number_of_sentences: int = Field(description="The total number of sentences in the email.")
    contains_placeholders: bool = Field(
        description="Indicates whether the email contains placeholders that need to be filled in.")

In [54]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Indicates whether the email is professional in tone and content.',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The total number of sentences in the email.',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Indicates whether the email contains placeholders that need to be filled in.',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [55]:
email = """
Hi [first_name]
I am writing to you to let you know that I am available for hire.
laters.
Regards,
[last_name]
"""

In [59]:
checker = Agent(name="checker", instructions="You review potential sales emails", model="gpt-5-mini",
                output_type=EmailReview)
result = await Runner.run(checker, email)

In [60]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)